## Idea here is to make scripts that will be capable of taking bulk csv in a normalized format that will then be able to run and automatically make data into a wanddb friendly format and standardise it


To be added:
- sql query generation at the end
- new compound deduplication (need to decide on the methadology -> fix below)
- sort out the doi collection cas right now new compounds just get none automatically (lotus did not have) (fix below)
- change the making working df for eventaul staging table section use an inner join the current implementation seems redundant by getting the compounds from wand that are in new compounds and then joining that 
- the check the error reporting system its not nca
## **Bulk compound normaliser**
#### **This compound normalizer should do the following:**
- get a list of compounds from wand and a list of new compounds
- check for missing inchis in new dropna and add to errors
- check new against wand and create a df that has all of the compounds not in wand and a df of compounds in wand
- **df of new compounds not in wand:**
    - generate canonical 2D and 3D SMILES from InChi with RDkit
    - fill na iupac using nispo
    - generate InChi key with RDkit
    - give wand compound_id starting after the last id from the wand compounds
    - assigns dois collections and cas as none and puts column names into db format
    - this df should be ready for a LOAD DATA LOCAL SQL
- **df of compounds in wand already:**
    - merges new compounds that are already in the df with the wand compounds that are in new compounds
    - adds synonyms from the new compounds to the existing synonyms with deduplication in a pipe seperated fromat
    - if existing wand name column name is empty incoming name fillsna
    - new name is removed if in synonyms
    - new iupac is added in a deduplicated manner to synonyms if there is an existing iupac
- **output:***
    - generate a text file with general info
    - an error file of compounds that had an issue during processing 
    - a new compounds csv that should be directly loadable into wand
    - a staging table for a bulk update

#### **Input requirements:**
- The new compounds have been deduplicated:
- columns with multiple entries are pipe ('|') seperated
- **column headings new compounds:**
    - id (use an index or whatever it is for errors to be identifiable to compound later and cannot be NA)
    - new_id (new_id column must be named with source then _id i.e. lotus_id, wiki_id, can have multiple)
    - inchi
    - n_name
    - n_synonyms
    - n_iupac
    - copyable:) ['id','new_id','inchi','n_name','n_synonyms','n_iupac']
- **column headings wand compounds:**
    - compound_id
    - inchi
    - name
    - synonyms
    - iupac

In [ ]:
# wand compounds expected headings ['compound_id','inchi','name','synonyms','iupac']
# new compounds expected headings ['id','new_id','inchi','n_name','n_synonyms','n_iupac'] 
# (new_id column must be named with source then _id i.e. lotus_id, wiki_id, can have multiple)
# all multi fields must be '|' seperated if have multi values
#//////////////////////////////////////////////////////inputs
wand_inchis_path = ''
new_compounds_path = ''
staging_table=''
error_output_path = ''
output_path = ''
report_path = ''
has_new_ids = True
sql_commands = ''
#//////////////////////////////////////////////////////imports
import pandas as pd
from rdkit import Chem
import time
import pubchempy as pcp
from nispo import mol_to_iupac
#//////////////////////////////////////////////////////data
wand_compounds = pd.read_csv(wand_inchis_path)
new_compounds = pd.read_csv(new_compounds_path)
origional_size = new_compounds.shape[0]
errors = pd.DataFrame(columns = ['id','type','error_msg'])
print('Files loaded', flush = True)
#//////////////////////////////////////////////////////functions
def get_smiles2D(inchi):
    global errors

    try:
        mol = Chem.MolFromInchi(inchi, treatWarningAsError=True)
        if mol:
            c_smiles = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=False)
            return c_smiles
        else:
            err = [inchi,'inchi','Could not convert inchi to 2D smiles']
            errors.loc[len(errors)] = err
            return None
    except Exception as e:
        err = [inchi,'inchi',str(e)]
        errors.loc[len(errors)] = err
        return None

def get_smiles3D(inchi):
    global errors
    try:
        mol = Chem.MolFromInchi(inchi, treatWarningAsError=True)
        if mol:
            i_smiles = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
            return i_smiles
        else:
            err = [inchi,'inchi','Could not convert inchi to isomericsmiles']
            errors.loc[len(errors)] = err
            return None
    except Exception as e:
        err = [inchi,'inchi',str(e)]
        errors.loc[len(errors)] = err
        return None

def get_cid(input,type):
    global errors
    for k in range(5):
        try:
            compound = pcp.get_compounds(input,type)
            if compound:
                cid = compound[0].cid
                return cid
            else:
                err = [input,type,'Does not appear to be in pubchem with this inchi']
                errors.loc[len(errors)] = err
                return None
        except Exception as e:
            error_msg = str(e)
            if '502' in error_msg or 'bad gateway' in error_msg.lower() or 'timeout' in error_msg.lower():
                time.sleep(0.2)
            else:
                err = [input,type,str(e)]
                errors.loc[len(errors)] = err
                return None
    err = [input,type,'retry not sucessful']
    errors.loc[len(errors)] = err
    return None

def get_inchikey(inchi):
    global errors
    try:
        mol = Chem.MolFromInchi(inchi, treatWarningAsError=True)
        if mol:
            key = Chem.MolToInchiKey(mol)
            return key
        else:
            err = [inchi,'inchi','Could not convert inchi to inchi key']
            errors.loc[len(errors)] = err
            return None
    except Exception as e:
        err = [inchi,'inchi',str(e)]
        errors.loc[len(errors)] = err
        return None

def get_iupac(inchi):
    global errors
    try:
        mol = Chem.MolFromInchi(inchi,treatWarningAsError=True)
        if mol:
            iupac = mol_to_iupac(mol)
            if pd.notna(iupac):
                return iupac
            else:
                err = [inchi, 'inchi', 'Could not generate IUPAC name']
                errors.loc[len(errors)] = err
                return None
        else:
          err = [inchi,'inchi','Could not convert inchi to iupac']
          errors.loc[len(errors)] = err
          return None  
    except Exception as e:
        err = [inchi,'inchi',str(e)]
        errors.loc[len(errors)] = err
        return None 

def synonym_deduplicator(old,new):
    old = '' if pd.isna(old) else str(old)
    new = '' if pd.isna(new) else str(new)
    old = old.split('|')
    new = new.split('|')
    old.remove('[]')
    new.remove('[]')
    old = [ol.strip() for ol in old if ol.strip()]
    new = [ne.strip() for ne in new if ne.strip()]

    old_set = set(old)
    old_set = {ol.lower() for ol in old_set}
    for ne in new:
        if ne and ne.lower() not in old_set:
            old.append(ne)
            old_set.add(ne.lower())
    return '|'.join(old)

def name_syn_remover(name,synonyms):
    name = '' if pd.isna(name) else str(name)
    synonyms = '' if pd.isna(synonyms) else str(synonyms)
    name = name.strip()
    synonyms = synonyms.split('|')
    synonyms = [s.strip() for s in synonyms if s.strip()]
    synonyms = [s for s in synonyms if s.lower() != name.lower()]
    return '|'.join(synonyms)

def cid_with_progress(series,function):
    tot = len(series)
    count = 0
    def wrapped(value):
        nonlocal count
        count += 1
        if count % 100==0 or count == tot or count == 1:
            print(f'CID progress: {count} of {tot} at {count/tot*100:.2f}%', flush=True)
        result = function(value)
        return result
    return series.apply(wrapped)
#//////////////////////////////////////////////////////new compounds
                        #//////////////////////////////check if inchis are missing or if are already in the DB and remove from new add to err and add as new metadata
wand_set = set(wand_compounds['inchi'].tolist())

if new_compounds['inchi'].isna().any():
    missing_inchi = pd.DataFrame(columns=['id','type','error_msg'])
    missing_inchi['id'] = new_compounds['id'][new_compounds['inchi'].isna()]
    missing_inchi['type'] = 'inchi'
    missing_inchi['error_msg'] = 'missing an inchi'
    errors = pd.concat([errors,missing_inchi], ignore_index=True)
new_compounds.dropna(subset=['inchi'], inplace=True)

in_db = pd.DataFrame(columns=['id','type','error_msg'])
in_db['id'] = new_compounds[new_compounds['inchi'].isin(wand_set)]['inchi'].to_list()
in_db['type'] = 'inchi'
in_db['error_msg'] = 'inchi already in DB'
errors = pd.concat([errors,in_db], ignore_index=True)
compounds_already_in_db = new_compounds[new_compounds['inchi'].isin(wand_set)].copy()
new_compounds = new_compounds[~new_compounds['inchi'].isin(wand_set)].copy()

print('Seperated onld and new', flush=True)
                        #//////////////////////////////add standardized smiles
print('Standardising smiles', flush=True)
new_compounds['canonical_smiles'] = new_compounds['inchi'].apply(get_smiles2D)
new_compounds['isomeric_smiles'] = new_compounds['inchi'].apply(get_smiles3D)
print('Standardised smiles', flush=True)
                        #//////////////////////////////add cid
#print('Adding cid', flush=True)
#new_compounds['cid'] = cid_with_progress(new_compounds['inchi'],lambda inchi: get_cid(inchi, 'inchi'))
#print('cids added', flush=True)
                        #//////////////////////////////Add iupac
print('Adding iupac', flush=True)
new_compounds['n_iupac'] = new_compounds['n_iupac'].fillna(new_compounds['inchi'].apply(get_iupac))
print('iupac added', flush=True)
                        #//////////////////////////////Add inchikey
print('Adding inchikey', flush=True)
new_compounds['inchi_key'] = new_compounds['inchi'].apply(get_inchikey)
print('DB formatting', flush=True)
                        #//////////////////////////////wand_ids
start_id = pd.to_numeric(wand_compounds['compound_id']).max()+1
new_compounds['compound_id'] = range(start_id, start_id+len(new_compounds))
                        #//////////////////////////////correct columns
new_compounds['dois'] = None
new_compounds['collections'] = None
new_compounds['coconut_id'] = None
new_compounds['cas'] = None
new_compounds = new_compounds.rename(columns={'n_name': 'name','n_synonyms': 'synonyms','n_iupac': 'iupac'})
new_compounds = new_compounds[['compound_id', 'name', 'inchi', 'inchi_key', 'canonical_smiles', 'isomeric_smiles', 'iupac', 'synonyms', 'collections', 'dois', 'coconut_id', 'cid', 'cas']]
print('New Compounds formatted and done', flush=True)
#//////////////////////////////////////////////////////update existing
print('Updating existing', flush=True)
                        #//////////////////////////////making working df for eventaul staging table
already_compounds = set(compounds_already_in_db['inchi'].to_list())
staging_df = wand_compounds.loc[wand_compounds['inchi'].isin(already_compounds)].copy()
                        #//////////////////////////////updating names,iupac and synonyms
staging_df = staging_df.merge(compounds_already_in_db, how='left',on='inchi')

staging_df['synonyms'] = staging_df.apply(lambda row: synonym_deduplicator(row['synonyms'],row['n_synonyms']),axis=1)
staging_df.drop(columns =['n_synonyms'], inplace=True)

staging_df['name'] = staging_df['name'].fillna(staging_df['n_name'])
staging_df['synonyms'] = staging_df.apply(lambda row: synonym_deduplicator(row['synonyms'],row['n_name']), axis=1)

staging_df['synonyms'] = staging_df.apply(lambda row: name_syn_remover(row['name'],row['synonyms']), axis=1)
staging_df.drop(columns = ['n_name'], inplace=True)

staging_df['iupac'] = staging_df['iupac'].fillna(staging_df['n_iupac'])
staging_df['synonyms'] = staging_df.apply(lambda row: synonym_deduplicator(row['synonyms'],row['iupac']), axis=1)
staging_df.drop(columns = ['n_iupac'], inplace=True)
print('Updated existing', flush=True)
#//////////////////////////////////////////////////////report
print('Generating report', flush=True)
new_size = new_compounds.shape[0]
errors_size = errors.shape[0]

items_removed = origional_size - new_size

error_types = errors['error_msg'].value_counts(dropna=False)

item = f"Compounds in starting file: {origional_size}\n\
         New compounds standardised to add to the database: {new_size}\n\
         Removed compounds: {items_removed}\n\
         Errors: {errors_size}\n\
         Error types:\n\
         {error_types}\n\
         "

with open(report_path, 'w') as f:
    f.write(item)
#//////////////////////////////////////////////////////output
print('Saving', flush=True)
new_compounds.to_csv(output_path, index=False) 
staging_df.to_csv(staging_table, index=False)
errors.to_csv(error_output_path, index=False)

# CID getter

In [ ]:
#!/bin/python3
# column names needed:['og_id','inchi','inchi_key','smiles','cid'] (can have the original ID it is not used) 
# writes names to read file
#module load python-3.9.5
#module load pubchempy
#//////////////////////////////////////////////////input
compounds_file = ''
err_out = ''
results_out = '' # txt for the stats of the run
#//////////////////////////////////////////////////imports
import time
import pandas as pd
import pubchempy as pcp
#//////////////////////////////////////////////////functions
def get_cid(input,type):
    if pd.isna(input):
        return f'not input ({type})'
    for k in range(5):
        try:
            compound = pcp.get_compounds(input,type)
            if compound:
                cid = compound[0].cid
                return cid
            else:
                return None
        except Exception as e:
            error_msg = str(e)
            if '502' in error_msg or 'bad gateway' in error_msg.lower() or 'timeout' in error_msg.lower():
                time.sleep(5)
            else:
                return str(e)
    return 'retry not sucessful'
       
def progress_bar(count, total):
    bar_length = 40
    filled_length = int(bar_length * count // total)
    bar = '█' * filled_length + '-' * (bar_length - filled_length)
    out = f'\rProgress: |{bar}| {count}/{total} ({(count/total)*100:.2f}%)'
    return out
#///////////////////////////////////////////////main
df = pd.read_csv(compounds_file)

err = []
count = 1
length = len(df)

for idx, row in df.iterrows():
    if pd.isna(row.cid):
        continue
    inchi = row['inchi']
    inchi_key = row['inchi_key']
    smiles = row['smiles']

    cid = get_cid(inchi,'inchi')

    if isinstance(cid,int):
        df.at[idx, 'cid'] = cid
    else:
        err.append([inchi,cid])
        cid = get_cid(smiles,'smiles')
        if isinstance(cid,int):
            df.at[idx, 'cid'] = cid
        else:
            err.append([inchi,cid])
            cid = get_cid(inchi_key,'inchikey')
            if isinstance(cid,int):
                df.at[idx, 'cid'] = cid
            else:
                err.append([inchi,cid])
                df.at[idx, 'cid'] = pd.NA
    time.sleep(0.2)
    count += 1

    if count % 100 == 0 or count == 1 or count == len(df):
        prog = progress_bar(count, length)
        print(prog, flush=True)

    if count % 1000 == 0 or count == 1 or count == len(df):
        df.to_csv(compounds_file, index=False)
        error_df = pd.DataFrame(err, columns=['inchi', 'error'])
        error_df.to_csv(err_out,index=False)

#////////////////////////////////////////////////////outputs
df.to_csv(compounds_file,index=False)
error_df = pd.DataFrame(err, columns=['inchi', 'error'])
error_df.to_csv(err_out,index=False)

total = df.shape[0]
cids = df['cid'].notna().sum()
nothing = df['cid'].isna().sum()

errrs = error_df.shape[0]
messages = error_df['error'].value_counts(dropna=False).to_string()

results = f""" 
Total compounds: {total:,}
Compounds returned with cid: {cids:,}
Compounds no result: {nothing:,}
Percent sucess: {cids/total*100:.2f}% 
Total errors: {errrs:,} 
Messages: {messages}"""

with open(results_out, 'w') as f:
    f.write(results)

only accepting full double barrel scientific names and variants no only genus or only species names
## **Bulk plant normalizer**
#### **This plant normalizer should do the following:**
- take in and format the wcvp
- take in a file retured from the checklist bank that has been deduplicated on the origional name
- keep only the GBIF name, ID, accepted name, accepted ID
- remove non-plants
- remove any genus or species only names (need both genus and species minimum)
- adding a cleaned name and matching the datasets based on that 
- fuzzy matching on the cleaned names using the fuzz.ratio scorer
- linking back to accepted ID
- collapsing into unified main ID with all else going to synonyms

In [ ]:
# the plants must go through the https://www.checklistbank.org/tools/name-match and that resulting file will be the new

# ///////////////////////////////////////////////////////inputs
wcvp_file = ''
new_checklisted_file = ''
staging_table= ''
error_output_path = ''
output_path = ''
report_path = ''
#////////////////////////////////////////////////////////imports
import pandas as pd
from rapidfuzz import process
#////////////////////////////////////////////////////////functions
def name_cleaner(name):
    if pd.isna(name):
        return pd.NA
    name = name.strip().lower().replace('-','').split()
    name = ' '.join(name)    
    return name
def is_incomplete_name(name):
    if pd.isna(name) or name == '' or not name.strip():
        return True
    else:
        name = name.strip()
        if len(name.split()) < 2:
            return True
        else: 
            return False
def fuzzy_match(og_name, og_id, name_id_dict):
    cutoff = 95
    result = process.extractOne(og_name,name_id_dict.keys())
    if result is None:
        err = err = [og_id, 'match error', f'Failed direct match and fuzzy match with {og_name} matching to none of the wcvp']
        errors.loc[len(errors)] = err
        return pd.NA
    match, chance, _ = result
    if chance < cutoff:
        err = [og_id, 'match error', f'Failed direct match and fuzzy match with {og_name} matching to {match} at {chance}% chance']
        errors.loc[len(errors)] = err
        return pd.NA
    else:
        return name_id_dict.get(match)
    

#////////////////////////////////////////////////////////data
                                #////////////////////////load data
error_messages =''
wcvp = pd.read_csv(wcvp_file, sep='|')
new = pd.read_csv(new_checklisted_file)
errors = pd.DataFrame(columns=['id','error_stage','exact_error'])
                                #////////////////////////format wcvp
wcvp = wcvp.rename(columns={'scientfiicname':'scientific_name','scientfiicnameauthorship':'scientific_name_authorship'})
wcvp.drop(columns=['genus','specificepithet','infraspecificepithet','taxonrank','taxonomicstatus','nomenclaturalstatus','taxonremarks'],inplace=True)

wcvp_name_check = wcvp[['scientific_name','taxonid']]
wcvp_name_check['clean_name'] = wcvp_name_check['scientific_name'].apply(name_cleaner)
wcvp_name_check_dic = {row.clean_name: row.taxonid for row in wcvp_name_check.itertuples()}
#////////////////////////////////////////////////////////normalize new plants
                                #////////////////////////drop unnecessary columns
new = new[['original_id','original_scientificName','matchType','matchIssues','ID','sceintificName','acceptedID','acceptedScientificName','kingdom']]
                                #////////////////////////removing non-plants
new = new.loc[new['kingdom'].eq('Plantae')].copy()
                                #////////////////////////removing incomplete names
new = new.loc[~new['original_scientificName'].apply(is_incomplete_name)].copy()
                                #////////////////////////adding clean name for direct matching
new['clean_name'] = new['original_scientificName'].apply(name_cleaner)
                                #////////////////////////checking for duplicates on clean name and removing if necessary
if new.duplicated(subset='clean_name').any():
   eagfdhskagfkhjasdgfkaj error_messages = error_messages + 'Once the names were cleaned for direct original name wcvp matching some of the cleaned names were duplicated the script keeps the first and removes the rest of the duplicates. Please ensure the original file and the checklist return file are deduplicated on the plant name. I this was done then the actual cleaning process leads to duplicates of names by removing "-" and whitespace as well as comparing only lowercase. This means redundancy was likely removed. The removed entries are in the error file. \n'
   eagfdhskagfkhjasdgfkaj duplicated = new.loc[new.duplicated(subset='clean_name',keep=False)].copy()
   eagfdhskagfkhjasdgfkaj duplicated_not_removed = duplicated.loc[duplicated.duplicated(subset='clean_name',keep='first')]['clean_name'].to_list()
   eagfdhskagfkhjasdgfkaj duplicated = duplicated.loc[~duplicated['clean_name'].isin(duplicated_not_removed)].copy()
   eagfdhskagfkhjasdgfkaj err = [[row.original_id,'Duplication',f'{row.original_name} cleaned to {row.clean_name} was duplicated after cleaning'] for row in duplicated.itertuples()]
   eagfdhskagfkhjasdgfkaj errors.loc[len(errors)] = err
   eagfdhskagfkhjasdgfkaj new = new.drop_duplicates(subset='clean_name',keep='first')
                                #////////////////////////matching cleaned names
new = new.merge(wcvp_name_check, how='left',on='clean_name', validate='many_to_one')
                                #////////////////////////fuzzy matching what is left
unmatched = new.loc[new['taxonid'].isna()].copy()
new['taxonid_fuzzy'] = unmatched.apply(lambda row: fuzzy_match(row.clean_name, row.original_id, wcvp_name_check_dic),axis=1)
                                #////////////////////////recording match numbers
new['match_type'] = pd.NA
new.loc[new['taxonid'].notna(),'match_type'] = 'direct_match'
new.loc[new['taxonid_fuzzy'].notna(),'match_type'] = 'fuzzy match' 
                                #////////////////////////removing all unmatched
#there is no need for an error handler here to add the dropped as they were added already during fuzzy match failure
new.dropna(subset='match_type',inplace=True)
                                #////////////////////////using taxon id to get no accepted info

                                #////////////////////////using accepted usage name id to get accepted info

                                #////////////////////////putting row non accepted into gbif and powo synonyms

                                #////////////////////////taking every accepted id that is taxon id and adding synonyms from where it is listed as accepted usage name id  

                                #////////////////////////formatting for DB with plant_id and correct columns

                                #remove entities that are weird matches
#////////////////////////////////////////////////////////plants already in db

#////////////////////////////////////////////////////////report
stats = f"""The number matched directly: {direct_matches:,} \n
            The number fuzzy matched: {fuzzy_matches:,}"""

item = error_messages + '\n' + stats
with open(report_path, 'w') as f:
    f.write(item)

#////////////////////////////////////////////////////////output